In [3]:
##### Converts raw raster on electricity consumption into final varaiables needed 
# pixel and subnational (vector) scale
# variables 
    # total consumption

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject, calculate_default_transform
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats

In [5]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# rasters
electricity = f"{cd}/Data/Raw/Predictors/Electricity_Chen/2019/EC2019.tif"

# save paths
electricity_binary = f"{cd}/Data/Raw/Predictors/Electricity_Chen/2019_binary.tif"

pixels = f"{cd}/Data/Clean/Predictors/Rasters/share_electrified.tif"
vectors = f"{cd}/Data/Clean/Predictors/Vectors/share_electrified.csv"

In [11]:
#### Electricity share of pixels at 5km resolution

### PREP
# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()


### STEP 1 - binary raster at 1km
# convert electricity raster to 0/1 (any non-zero value = has electricity)
with rasterio.open(electricity) as src:
    elec_data   = src.read(1).astype(np.float32)
    binary_meta = src.meta.copy()

# anything < 0 is nodata, 0 stays 0, anything > 0 becomes 1
elec_binary = np.where(elec_data > 10, 1.0, 0.0)

binary_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

with rasterio.open(electricity_binary, "w", **binary_meta) as dst:
    dst.write(elec_binary, 1)

### STEP 2 - resample to 5km
# average of 0/1 binary = share of pixels with electricity
def resample_to_ref(src_path):
    dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            dst_crs=dst_crs,
            dst_transform=dst_transform,
            resampling=Resampling.average,
            src_nodata=-9999,
            dst_nodata=np.nan,
        )
    dst_array[dst_array == -9999] = np.nan
    return dst_array

elec_share = resample_to_ref(electricity_binary)
elec_share  = np.clip(elec_share, 0, 1)


### SAVE
out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = elec_share.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels, "w", **out_meta) as dst:
    dst.write(out_arr, 1)

In [13]:
with rasterio.open(electricity) as src:
    elec_data   = src.read(1).astype(np.float32)
    binary_meta = src.meta.copy()

# anything < 0 is nodata, 0 stays 0, anything > 0 becomes 1
elec_binary = np.where(elec_data < 0, -9999, np.where(elec_data > 0, 1.0, 0.0))

# verify before writing
print(f"number of 0s in binary:     {(elec_binary == 0).sum():,}")
print(f"number of 1s in binary:     {(elec_binary == 1).sum():,}")
print(f"number of nodatas in binary: {(elec_binary == -9999).sum():,}")

binary_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

with rasterio.open(electricity_binary, "w", **binary_meta) as dst:
    dst.write(elec_binary, 1)

number of 0s in binary:     13,110,219
number of 1s in binary:     119,963,100
number of nodatas in binary: 426,780,041
